# Receptive Fields across Multiple Depths

This notebook characterizes receptive fields (RFs) and spatial retinotopy measured under multi-depth visual stimulation (`SpheresPermTubeReward_multidepth`), providing the full set of single-session and population figures:

### Panels:
- **A**: Model formulation schematic ($\Delta F/F_0 = S\mathbf{b}$)
- **B**: Reconstructed visual stimulus frames across 8 depths (5 to 640 cm) under multi-depth stimulation
- **C**: Depth tuning curves and multi-depth RF maps for 3 example V1 neurons (ROIs 53, 118, 623 of `PZAH17.1e_S20250318`)
- **D**: 3D receptive field volumes rendered as nested isosurfaces (50-98% of peak, colour and opacity ramping to the core), on a clean white background with faint floor/wall marginal projections
- **E**: Population correlation: Preferred receptive field depth vs Preferred trial-average depth
- **F, G, H**: Spatial retinotopic organization of an example V1 FOV (`PZAG16.3c_S20250317`):
  - **F**: Preferred Azimuth (degrees)
  - **G**: Preferred Elevation (degrees)
  - **H**: Preferred Virtual Depth (cm) with anatomical 2P mean image inset
- **Full Figure**: Unified publication canvas assembling Panels A–H

In [ ]:
%reload_ext autoreload
%autoreload 2

import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import spheres
from cottage_analysis.analysis.spheres import rf_analysis
from cottage_analysis.plotting import rf_plots, depth_selectivity_plots, rsof_plots, plotting_utils, style
from cottage_analysis.pipelines import pipeline_utils
from v1_depth_map.figure_utils import receptive_fields as rf_utils
from v1_depth_map.revisions.revision_sessions import sessions
from v1_depth_map.paths import get_figures_roots

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

## Load Population Data


In [ ]:
# Load all multi-depth sessions for population correlation
multi_depth_sessions = [k for k, v in sessions.items() if v == "multidepth"]

all_sig_m, all_sig_ipsi_m, neurons_df = rf_analysis.load_sig_rf(
    flexilims_session=flexilims_session,
    session_list=multi_depth_sessions,
    n_std=6,
    filter_datasets={"annotated": True},
    use_multidepth=True,
    use_cols=None,
    sphere_presentation_mask=None,
)

select_neurons = (neurons_df["iscell"] == 1) & (
    depth_selectivity_plots.common_utils.one_sided_pval_from_spearman(
        neurons_df["depth_tuning_test_spearmanr_rval_closedloop"],
        neurons_df["depth_tuning_test_spearmanr_pval_closedloop"],
    )
    < 0.05
)
neurons_df["is_depth_selective"] = select_neurons
depth_neurons = neurons_df[select_neurons & (neurons_df["rf_sig"] == 1)].copy()
print(
    f"Loaded {len(depth_neurons)} depth-selective neurons with significant RFs across {len(multi_depth_sessions)} sessions."
)

## Load Example Session Data


In [ ]:
# Load example session data for stimulus frames, tuning curves, and receptive fields
example_session = "PZAH17.1e_S20250318"

frames_rf, imaging_df_rf = spheres.regenerate_frames_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward_multidepth",
    photodiode_protocol=5,
    return_volumes=True,
    resolution=1,
    verbose=False,
    is_multidepth=True,
)
plt.close("all")

neurons_df_example = pd.read_pickle(
    pipeline_utils.create_neurons_ds(
        session_name=example_session,
        flexilims_session=flexilims_session,
        project=None,
        conflicts="skip",
    ).path_full
)

_, trials_df_example = spheres.sync_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)

# Load suite2p ROI metadata for FOV spatial plots
suite2p_ds = flz.get_datasets(
    flexilims_session=flexilims_session,
    origin_name=example_session,
    dataset_type="suite2p_rois",
    filter_datasets={"anatomical_only": 3, "annotated": True},
    allow_multiple=False,
    return_dataseries=False,
)
iscell = np.load(suite2p_ds.path_full / "plane0" / "iscell.npy", allow_pickle=True)[
    :, 0
]
neurons_df_example["iscell"] = iscell
stat = np.load(suite2p_ds.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops = np.load(suite2p_ds.path_full / "plane0" / "ops.npy", allow_pickle=True).item()

In [ ]:
# Load a second session for the FOV retinotopy panels (E, F, G): the example
# neurons come from `example_session`, but this FOV has the cleaner gradient
fov_session = "PZAG16.3c_S20250317"

neurons_df_fov = pd.read_pickle(
    pipeline_utils.create_neurons_ds(
        session_name=fov_session,
        flexilims_session=flexilims_session,
        project=None,
        conflicts="skip",
    ).path_full
)

suite2p_ds_fov = flz.get_datasets(
    flexilims_session=flexilims_session,
    origin_name=fov_session,
    dataset_type="suite2p_rois",
    filter_datasets={"anatomical_only": 3, "annotated": True},
    allow_multiple=False,
    return_dataseries=False,
)
neurons_df_fov["iscell"] = np.load(
    suite2p_ds_fov.path_full / "plane0" / "iscell.npy", allow_pickle=True
)[:, 0]
stat_fov = np.load(suite2p_ds_fov.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops_fov = np.load(
    suite2p_ds_fov.path_full / "plane0" / "ops.npy", allow_pickle=True
).item()

## Shared Settings & Panel Helpers

Every panel is drawn by a function that takes the target figure (or axes) and the
axes rectangles, so the same code produces the standalone panels below and the
assembled figure at the end of the notebook. Styling knobs (example neurons,
colormaps, isosurface levels, marginal opacity) live at the top of the cell.


In [ ]:
# ------------------------------------------------------------------ constants
# Example neurons: near / middle / far depth-tuned, with distinct azimuth and
# elevation tuning (session PZAH17.1e_S20250318)
SELECT_ROIS = [53, 118, 623]
# One sequential colormap per example neuron, shared by Panels C and D
EXAMPLE_CMAPS = ["Purples", "Greens", "Oranges"]
EXAMPLE_COLORS = [plt.get_cmap(name)(0.85) for name in EXAMPLE_CMAPS]

FOV_WIDTH = 661


# -------------------------------------------------------------- layout helpers
# Centimetre-based layout helpers, shared so panel letters stay at
# FONTSIZE_DICT["panel"] across every figure.
from cottage_analysis.plotting.style import rect_cm, panel_letter

# NOTE: when passing a rect to `plot_rf` or `plot_stimulus_frame`, `y` is the
# *top* of the stack of per-depth axes and `h` its total height, since those
# helpers grow downwards.

# ------------------------------------------------------------ Panels B, C, E-G
def plot_panel_b(fig, position, frames_contra, start_frame, depths, fontsize_dict=None):
    """Panel B: reconstructed stimulus frame at each of the 8 depths."""
    plt.figure(fig.number)
    rf_plots.plot_stimulus_frame(
        frame=frames_contra[:, start_frame],
        idepth=None,
        depths=depths,
        position=position,
        fontsize_dict=fontsize_dict or FONTSIZE_DICT,
        plot_prop=0.9,
    )


def plot_panel_c(
    fig,
    neurons_df,
    trials_df,
    tuning_rect,
    rf_rect,
    dx,
    rois=None,
    colors=None,
    ndepths=8,
    fontsize_dict=None,
):
    """Panel C: depth tuning curve + multi-depth RF map per example neuron.

    `tuning_rect` and `rf_rect` are the rects of the leftmost neuron; each
    subsequent neuron is shifted right by `dx` (figure fractions throughout).
    """
    fig = plt.figure(fig.number)
    rois = SELECT_ROIS if rois is None else rois
    colors = EXAMPLE_COLORS if colors is None else colors
    fontsize_dict = fontsize_dict or FONTSIZE_DICT
    axes = []
    for iroi, (roi, linecolor) in enumerate(zip(rois, colors)):
        ax = fig.add_axes([tuning_rect[0] + dx * iroi, *tuning_rect[1:]])
        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_df,
            trials_df=trials_df,
            roi=roi,
            linecolor=linecolor,
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1,
            closed_loop=1,
            fontsize_dict=fontsize_dict,
            markersize=5,
            markeredgecolor="w",
            ylim_precision_base=5,
            ylim_precision=2,
        )
        ax.set_ylabel("")
        if iroi == len(rois) // 2:
            ax.set_xlabel("Virtual depth (cm)", fontsize=fontsize_dict["label"])
        else:
            ax.set_xlabel("")
        # The tuning curve ticks every depth; keep every other one to declutter
        xticks = ax.get_xticks()
        xlabels = [t.get_text() for t in ax.get_xticklabels()]
        ax.set_xticks(
            xticks[::2], xlabels[::2], rotation=45, fontsize=fontsize_dict["tick"]
        )
        ax.tick_params(length=1.5)
        axes.append(ax)
        rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=[rf_rect[0] + dx * iroi, *rf_rect[1:]],
            plot_prop=0.9,
            xlabel="Azimuth (degrees)" if iroi == len(rois) // 2 else "",
            ylabel="Elevation (degrees)" if iroi == 0 else "",
            fontsize_dict=fontsize_dict,
            use_multidepth=True,
            plot_yticklabels=iroi == 0,
        )
        
    return axes


def plot_panel_c_population(fig, depth_neurons, rect, fontsize_dict=None):
    """Panel C inset: RF preferred depth vs trial-averaged preferred depth."""
    rsof_plots.plot_2d_hist(
        fig=fig,
        neurons_df=depth_neurons,
        ycol="rf_preferred_depth_closedloop_multidepth",
        xcol="preferred_depth_closedloop",
        ylabel="Preferred receptive\nfield depth (cm)",
        xlabel="Preferred trial-average\ndepth (cm)",
        color="k",
        contour_color="gray",
        plot_x=rect[0],
        plot_y=rect[1],
        plot_width=rect[2],
        plot_height=rect[3],
        s=5,
        alpha=0.5,
        plot_diagonal=True,
        aspect_equal=True,
        fontsize_dict=fontsize_dict or FONTSIZE_DICT,
    )


def plot_panels_efg(
    fig,
    rects,
    neurons_df_fov,
    stat_fov,
    ops_fov,
    depths,
    inset_rect=None,
    fov_width=FOV_WIDTH,
    fontsize_dict=None,
):
    """Panels E, F, G: preferred azimuth / elevation / depth across the FOV.

    `rects` is the three axes rects; `inset_rect` places the anatomical meanImg
    inset inside Panel G (skipped when None).
    """
    plt.figure(fig.number)
    fontsize_dict = fontsize_dict or FONTSIZE_DICT
    maps = [
        ("rf_azi", cm.YlOrRd.reversed()),
        ("rf_ele", cm.YlOrRd.reversed()),
        ("preferred_depth_closedloop", cm.cool.reversed()),
    ]
    images = []
    for rect, (col, cmap) in zip(rects, maps):
        fig.add_axes(rect)
        images.append(
            depth_selectivity_plots.plot_example_fov(
                neurons_df=neurons_df_fov,
                ops=ops_fov,
                stat=stat_fov,
                ndepths=len(depths),
                col=col,
                cmap=cmap,
                background_color=np.array([0, 0, 0]),
                fontsize_dict=fontsize_dict,
                fov_width=fov_width,
            )
        )
    if inset_rect is not None:
        plotting_utils.plot_white_rectangle(
            inset_rect[0] - 0.005,
            inset_rect[1] - 0.005,
            inset_rect[2] + 0.01,
            inset_rect[3] + 0.01,
        )
        fig.add_axes(inset_rect)
        depth_selectivity_plots.plot_fov_mean_img(
            ops_fov["meanImg"], fov_width=fov_width, vmax=4000
        )
    return images


# ------------------------------------------------------------ Panel D (3D RFs)
# Drawing helpers live in v1_depth_map.figure_utils.receptive_fields (rf_utils)
def rf_volume(neurons_df, roi, ndepths, azi_slice, frame_shape=(16, 24)):
    """Fold-averaged, rectified multi-depth RF volume, cropped in azimuth."""
    coef = neurons_df.loc[roi, "rf_coef_closedloop_multidepth"][:, :-1]
    coef = coef.reshape(coef.shape[0], ndepths, *frame_shape)[..., azi_slice]
    return np.maximum(np.mean(coef, axis=0), 0)


def plot_panel_d(
    ax,
    neurons_df,
    depths_arr,
    rois=None,
    cmaps=None,
    colors=None,
    fontsize_dict=None,
    elev=28,
    azim=-50,
    tickpad=-2,
    labelpad=-10,
):
    """Panel D: 3D receptive-field volumes of the example neurons in visual space.

    The drawing (isosurface stack, peak marker, marginal projections, cube
    styling) and the displayed window live in
    `v1_depth_map.figure_utils.receptive_fields`.

    `tickpad` (cube to tick labels) and `labelpad` (tick labels to axis labels)
    are forwarded to `rf_utils.style_rf_3d_axes`; both are negative because
    mplot3d spaces them generously for a panel this small.
    """
    rois = SELECT_ROIS if rois is None else rois
    cmaps = EXAMPLE_CMAPS if cmaps is None else cmaps
    colors = EXAMPLE_COLORS if colors is None else colors
    azi_slice = slice(
        int(rf_utils.AZI_LIM[0] / rf_utils.AZI_RESOLUTION),
        int(rf_utils.AZI_LIM[1] / rf_utils.AZI_RESOLUTION),
    )
    for roi, cmap_name, color in zip(rois, cmaps, colors):
        volume = rf_volume(neurons_df, roi, len(depths_arr), azi_slice)
        rf_utils.add_rf_isosurfaces(ax, volume, cmap_name)
        peak = rf_utils.add_rf_peak_marker(ax, volume, color)
        rf_utils.add_rf_marginals(ax, volume, color, peak)
    rf_utils.style_rf_3d_axes(
        ax, depths_arr, fontsize_dict, elev=elev, azim=azim, tickpad=tickpad, labelpad=labelpad
    )
    return ax

# Aliases for panel re-lettering
plot_panel_e_population = plot_panel_c_population
plot_panels_fgh = plot_panels_efg

## Panels B & C: Multi-Depth Stimuli & Example Neuron RFs


In [ ]:
depths = sorted(neurons_df.best_depth.unique())
ndepths = len(depths)

# A frame where spheres are present at every depth
frames_contra = frames_rf[..., frames_rf.shape[-1] // 2 :]
has_data = np.any(frames_contra, axis=(2, 3))
start_frame = np.where(np.sum(has_data, axis=0) == ndepths)[0][2450]

fig = plt.figure(figsize=(18 / 2.54, 18 / 2.54))
plot_panel_b(fig, [0, 0.74, 0.3, 0.4], frames_contra, start_frame, depths)
axes = plot_panel_c(
    fig,
    neurons_df=neurons_df_example,
    trials_df=trials_df_example,
    tuning_rect=[0.267, 0.9, 0.07, 0.05],
    rf_rect=[0.055, 0.8, 0.5, 0.5],
    dx=0.13,
    ndepths=ndepths,
)
plot_panel_c_population(fig, depth_neurons, [0.68, 0.74, 0.25, 0.21])

style.savefig(
    SAVE_ROOT / "fig_receptive_fields_examples.svg", bbox_inches="tight", fig=fig
)
plt.show()

## Exploratory: Plotly Volume Renderer (not used in the assembled figure)


In [ ]:
# Plot 3D spatio-temporal receptive fields in visual space (Plotly volume)
depths = sorted(neurons_df.best_depth.unique())
rf_plots.plot_rf_3d(
    neurons_df_example,
    SELECT_ROIS,
    depths,
    SAVE_ROOT / "fig_receptive_fields_3d_rfs.pdf",
    dict(label=10, tick=10),
)

## Panel D: 3D Receptive Fields (Nested Isosurfaces)

This polished version matches the original viewing perspective (Azimuth on the left-front, Depth on the right-front, Elevation vertical) while using 12 nested isosurfaces spanning 50-98% of peak, with colour and opacity ramping from a pale envelope to a dark core on a clean white background with publication lighting, drop lines, and 2D marginal projections on the floor (Azimuth $\times$ Depth) and walls.


In [ ]:
depths_arr = np.array(sorted(neurons_df.best_depth.unique()))

fig = plt.figure(figsize=(9 / 2.54, 8 / 2.54), dpi=300)
fig.patch.set_facecolor("white")
plot_panel_d(fig.add_subplot(111, projection="3d"), neurons_df_example, depths_arr,
             elev=38, azim=-40, tickpad=-2, labelpad=-5
)

for suffix in ("svg", "pdf"):
    style.savefig(
        SAVE_ROOT / f"fig_receptive_fields_3d_rfs_polished.{suffix}",
        bbox_inches="tight",
        fig=fig,
        dpi=600,  # resolution of the rasterised volumes
    )
plt.show()

## Panels F, G, H: Spatial Retinotopy in V1 FOV

In [ ]:
depths = sorted(neurons_df.best_depth.unique())

fig = plt.figure(figsize=(18 / 2.54, 8 / 2.54), dpi=300)
plot_panels_fgh(
    fig,
    rects=[
        [0.04, 0.05, 0.28, 0.88],
        [0.37, 0.05, 0.28, 0.88],
        [0.70, 0.05, 0.28, 0.88],
    ],
    neurons_df_fov=neurons_df_fov,
    stat_fov=stat_fov,
    ops_fov=ops_fov,
    depths=depths,
    inset_rect=[0.86, 0.66, 0.08, 0.08],
)

style.savefig(SAVE_ROOT / "fig_receptive_fields_fov.svg", bbox_inches="tight", fig=fig)
plt.show()

## Full Figure: Unified Publication Layout (Panels A–H)

This cell compiles all individual panels into a single, cohesive publication figure canvas (18.5 × 17.0 cm):
- **A**: Model formulation schematic
- **B**: Multi-depth reconstructed visual stimulus frames across 8 depths
- **C**: Depth tuning curves & 8-depth RF heatmaps for 3 example V1 neurons
- **D**: 3D receptive fields as nested isosurfaces (50-98% of peak) + faint floor/wall marginal projections
- **E**: Population correlation (RF preferred depth vs trial-averaged preferred depth)
- **F, G, H**: Spatial retinotopy in example V1 FOV (Azimuth, Elevation, Preferred Depth + meanImg inset)

In [ ]:
# Assembled figure, 18.5 x 17.0 cm. All positions are in cm from the bottom-left
# corner of the canvas; `rect_cm` converts them to figure fractions. Panel A is
# left blank for the model schematic, which is dropped in afterwards.
FIG_W, FIG_H = 18.5, 17.0
LAYOUT = dict(
    letters={
        "A": (0.15, 16.6),
        "B": (0.15, 14.4),
        "C": (2.10, 16.6),
        "D": (7.60, 16.6),
        "E": (11.00, 10.6),
        "F": (0.15, 5.75),
        "G": (6.10, 5.75),
        "H": (12.05, 5.75),
    },
    stimulus=(0.75, 13.6, 1.25, 7.2),  # x, top row bottom, width, total height
    tuning=(2.75, 15.8, 1.20, 0.95),
    rf_maps=(2.70, 13.6, 1.25, 7.2),
    example_dx=1.55,
    volume=(7.5, 10.6, 10.6, 6.1),
    population=(12.0, 6.3, 5.2, 4.0),
    fov=[(0.35, 0.35, 5.75, 5.2), (6.30, 0.35, 5.75, 5.2), (12.25, 0.35, 5.75, 5.2)],
    fov_inset=(15.45, 3.65, 1.75, 1.75),
)

depths = sorted(neurons_df.best_depth.unique())
depths_arr = np.array(depths)
ndepths = len(depths)

frames_contra = frames_rf[..., frames_rf.shape[-1] // 2 :]
has_data = np.any(frames_contra, axis=(2, 3))
start_frame = np.where(np.sum(has_data, axis=0) == ndepths)[0][2450]

fig = plt.figure(figsize=(FIG_W / 2.54, FIG_H / 2.54), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xticks([])
ax.set_yticks([])
fig.patch.set_facecolor("white")
for letter, (x, y) in LAYOUT["letters"].items():
    panel_letter(fig, letter, x, y)

# B: multi-depth stimulus frames
plot_panel_b(fig, rect_cm(fig, *LAYOUT["stimulus"]), frames_contra, start_frame, depths)

# C: example neuron tuning curves and RF maps
axes = plot_panel_c(
    fig,
    neurons_df=neurons_df_example,
    trials_df=trials_df_example,
    tuning_rect=rect_cm(fig, *LAYOUT["tuning"]),
    rf_rect=rect_cm(fig, *LAYOUT["rf_maps"]),
    dx=LAYOUT["example_dx"] / FIG_W,
    ndepths=ndepths,
)

# E: Population correlation (scatter plot below 3D volume)
plot_panel_e_population(fig, depth_neurons, rect_cm(fig, *LAYOUT["population"]))

# D: 3D receptive-field volumes (to the right of example neurons)
plot_panel_d(
    fig.add_axes(rect_cm(fig, *LAYOUT["volume"]), projection="3d"),
    neurons_df_example,
    depths_arr,
    tickpad=-4,
    labelpad=-8,
    elev=40, 
    azim=-35,
)

# F, G, H: retinotopy across the example FOV
plot_panels_fgh(
    fig,
    rects=[rect_cm(fig, *r) for r in LAYOUT["fov"]],
    neurons_df_fov=neurons_df_fov,
    stat_fov=stat_fov,
    ops_fov=ops_fov,
    depths=depths,
    inset_rect=rect_cm(fig, *LAYOUT["fov_inset"]),
)

for suffix in ("svg", "pdf"):
    style.savefig(
        SAVE_ROOT / f"fig_receptive_fields_full.{suffix}",
        fig=fig,
        dpi=600,  # resolution of the rasterised 3D volumes
    )
plt.show()